# Delta Lake com Apache Spark (PySpark)

## Cenário: Plataforma de E-commerce

Neste notebook, demonstramos o uso do **Delta Lake** com **Apache Spark (PySpark)** aplicado a um sistema de pedidos de e-commerce.

O cenário modela três entidades:
- **Cliente** — dados cadastrais do comprador
- **Produto** — catálogo com categoria e preço
- **Pedido** — relaciona cliente e produto com status, quantidade e data

Demonstraremos as operações **DDL**, **INSERT**, **UPDATE**, **DELETE** e **Time Travel**.

## Modelo Entidade-Relacionamento

```
┌──────────────┐         ┌──────────────┐
│   CUSTOMER   │         │   PRODUCT    │
├──────────────┤         ├──────────────┤
│ customer_id  │PK       │ product_id   │PK
│ name         │         │ name         │
│ email        │         │ category     │
│ city         │         │ price        │
└──────┬───────┘         └──────┬───────┘
       │                        │
       └──────────┬─────────────┘
                  │
                  ▼
        ┌─────────────────────┐
        │        ORDER        │
        ├─────────────────────┤
        │ order_id     (PK)   │
        │ customer_id  (FK)   │
        │ product_id   (FK)   │
        │ quantity            │
        │ unit_price          │
        │ status              │
        │ order_date          │
        └─────────────────────┘
```

| Entidade | Atributos |
|---|---|
| CUSTOMER | customer_id (PK), name, email, city |
| PRODUCT  | product_id (PK), name, category, price |
| ORDER    | order_id (PK), customer_id (FK), product_id (FK), quantity, unit_price, status, order_date |

## 1. Configuração — SparkSession com Delta Lake

In [9]:
import sys
import os
# Adiciona o diretório src ao path para permitir importação do pacote local
sys.path.append(os.path.abspath("../src"))

from spark_delta_apache import get_spark_delta_session, setup_logging

# Configurar logging e criar SparkSession via pacote local
setup_logging()
spark = get_spark_delta_session("DeltaLakeEcommerce")
spark


## 2. DDL — Criação das Tabelas

Criamos as três tabelas com `USING delta`. O Spark armazena os dados em formato **Parquet** e mantém o histórico de transações na pasta `_delta_log/`.

In [2]:
# Limpar tabelas existentes para garantir execução limpa
spark.sql("DROP TABLE IF EXISTS customer_delta")
spark.sql("DROP TABLE IF EXISTS product_delta")
spark.sql("DROP TABLE IF EXISTS order_delta")

spark.sql("""
    CREATE TABLE IF NOT EXISTS customer_delta (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING
    ) USING delta
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS product_delta (
        product_id INT,
        name       STRING,
        category   STRING,
        price      DOUBLE
    ) USING delta
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS order_delta (
        order_id    INT,
        customer_id INT,
        product_id  INT,
        quantity    INT,
        unit_price  DOUBLE,
        status      STRING,
        order_date  STRING
    ) USING delta
""")

print("Tabelas criadas com sucesso!")
spark.sql("SHOW TABLES").show()

26/04/28 02:22:46 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

Tabelas criadas com sucesso!
+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|customer_delta|      false|
|  default|   order_delta|      false|
|  default| product_delta|      false|
+---------+--------------+-----------+



## 3. INSERT — Inserção de Dados

Cada `INSERT` gera uma nova **versão** no `_delta_log`, registrando quais arquivos Parquet foram adicionados.

In [3]:
# Inserir clientes
spark.sql("""
    INSERT INTO customer_delta VALUES
        (1, 'Ana Silva',   'ana@email.com',   'São Paulo'),
        (2, 'Bruno Costa', 'bruno@email.com', 'Rio de Janeiro'),
        (3, 'Carla Lima',  'carla@email.com', 'Curitiba')
""")

# Inserir produtos
spark.sql("""
    INSERT INTO product_delta VALUES
        (1, 'Notebook',      'Eletronicos', 3500.00),
        (2, 'Mouse',         'Eletronicos',   89.90),
        (3, 'Cadeira Gamer', 'Moveis',      1200.00)
""")

# Inserir pedidos
spark.sql("""
    INSERT INTO order_delta VALUES
        (1, 1, 1, 1, 3500.00, 'pendente', '2024-01-10'),
        (2, 2, 2, 2,   89.90, 'aprovado', '2024-01-11'),
        (3, 3, 3, 1, 1200.00, 'pendente', '2024-01-12')
""")

print("=== Clientes ===")
spark.sql("SELECT * FROM customer_delta").show()

print("=== Produtos ===")
spark.sql("SELECT * FROM product_delta").show()

print("=== Pedidos ===")
spark.sql("SELECT * FROM order_delta").show()

=== Clientes ===


26/04/28 02:22:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-----------+-----------+---------------+--------------+
|customer_id|       name|          email|          city|
+-----------+-----------+---------------+--------------+
|          2|Bruno Costa|bruno@email.com|Rio de Janeiro|
|          3| Carla Lima|carla@email.com|      Curitiba|
|          1|  Ana Silva|  ana@email.com|     São Paulo|
+-----------+-----------+---------------+--------------+

=== Produtos ===


+----------+-------------+-----------+------+
|product_id|         name|   category| price|
+----------+-------------+-----------+------+
|         1|     Notebook|Eletronicos|3500.0|
|         3|Cadeira Gamer|     Moveis|1200.0|
|         2|        Mouse|Eletronicos|  89.9|
+----------+-------------+-----------+------+

=== Pedidos ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       1|          1|         1|       1|    3500.0|pendente|2024-01-10|
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
+--------+-----------+----------+--------+----------+--------+----------+



## 4. UPDATE — Atualização de Dados

O Delta Lake implementa `UPDATE` como **copy-on-write**: novos arquivos Parquet são escritos com os dados atualizados e o `_delta_log` registra a mudança. Os arquivos originais permanecem intactos (disponíveis para Time Travel).

In [4]:
# Atualizar status do pedido 1 para 'aprovado'
spark.sql("""
    UPDATE order_delta
    SET status = 'aprovado'
    WHERE order_id = 1
""")

# Atualizar preço do Notebook com desconto
spark.sql("""
    UPDATE product_delta
    SET price = 3299.00
    WHERE product_id = 1
""")

print("=== Pedidos após UPDATE ===")
spark.sql("SELECT * FROM order_delta").show()

print("=== Produtos após UPDATE ===")
spark.sql("SELECT * FROM product_delta").show()

=== Pedidos após UPDATE ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       1|          1|         1|       1|    3500.0|aprovado|2024-01-10|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
+--------+-----------+----------+--------+----------+--------+----------+

=== Produtos após UPDATE ===
+----------+-------------+-----------+------+
|product_id|         name|   category| price|
+----------+-------------+-----------+------+
|         3|Cadeira Gamer|     Moveis|1200.0|
|         1|     Notebook|Eletronicos|3299.0|
|         2|        Mouse|Eletronicos|  89.9|
+----------+-------------+-----------+------+



## 5. DELETE — Remoção de Dados

O `DELETE` também é copy-on-write. Os dados removidos ainda existem nos arquivos físicos e podem ser recuperados via **Time Travel** até que `VACUUM` seja executado.

In [5]:
# Cancelar (remover) o pedido 3
spark.sql("""
    DELETE FROM order_delta
    WHERE order_id = 3
""")

print("=== Pedidos após DELETE ===")
spark.sql("SELECT * FROM order_delta").show()

=== Pedidos após DELETE ===


+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       1|          1|         1|       1|    3500.0|aprovado|2024-01-10|
+--------+-----------+----------+--------+----------+--------+----------+



## 6. Time Travel e Histórico de Transações

O Delta Lake mantém um **log de transações** em `_delta_log/`. Cada operação (CREATE, INSERT, UPDATE, DELETE) cria uma nova versão numerada, permitindo leitura de qualquer versão histórica.

In [6]:
from delta.tables import DeltaTable

# Visualizar histórico completo de transações
dt_orders = DeltaTable.forName(spark, "order_delta")
print("=== Histórico de Transações (order_delta) ===")
dt_orders.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

=== Histórico de Transações (order_delta) ===
+-------+-----------------------+------------+----------------------------------------------------------------------------------------------+
|version|timestamp              |operation   |operationParameters                                                                           |
+-------+-----------------------+------------+----------------------------------------------------------------------------------------------+
|3      |2026-04-28 02:23:15.422|DELETE      |{predicate -> ["(order_id#6050 = 3)"]}                                                        |
|2      |2026-04-28 02:23:09.401|UPDATE      |{predicate -> ["(order_id#3444 = 1)"]}                                                        |
|1      |2026-04-28 02:22:52.838|WRITE       |{mode -> Append, partitionBy -> []}                                                           |
|0      |2026-04-28 02:22:49.648|CREATE TABLE|{partitionBy -> [], clusterBy -> [], description -> NULL

In [7]:
# Time Travel: ler a versão 0 (estado após o CREATE TABLE, antes de qualquer INSERT)
print("=== Time Travel — Versão 0 (CREATE TABLE) ===")
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("order_delta") \
    .show()

# Time Travel: ler a versão 1 (após o primeiro INSERT)
print("=== Time Travel — Versão 1 (após INSERT) ===")
spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .table("order_delta") \
    .show()

# Estado atual (após UPDATE e DELETE)
print("=== Estado Atual ===")
spark.sql("SELECT * FROM order_delta").show()

=== Time Travel — Versão 0 (CREATE TABLE) ===


+--------+-----------+----------+--------+----------+------+----------+
|order_id|customer_id|product_id|quantity|unit_price|status|order_date|
+--------+-----------+----------+--------+----------+------+----------+
+--------+-----------+----------+--------+----------+------+----------+

=== Time Travel — Versão 1 (após INSERT) ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       1|          1|         1|       1|    3500.0|pendente|2024-01-10|
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
+--------+-----------+----------+--------+----------+--------+----------+

=== Estado Atual ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price

In [8]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
